In [3]:
import os
import torch 
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms

In [4]:
# device=torch.accelerator.current_accelerator.type() if torch.accelerator.is_available() else "cpu"
# print(f"using {device} device")
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device")

Using cpu device


In [5]:
import torch
import sys

print(f"Torch version: {torch.__version__}")
print(f"Python executable: {sys.executable}")

Torch version: 2.5.1
Python executable: C:\Users\YYY\miniconda3\envs\torch_env\python.exe


In [13]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__();
        self.flatten=nn.Flatten()
# 第二部分：定义“零件”（层）
# self.flatten = nn.Flatten()
# 含义：定义一个“展平”层。
# 动作：将输入的 28×28 二维图片压成一根长度为 784 的一维向量。
# 为什么？：因为后面的 nn.Linear（全连接层）只接收一维的输入。
        self.linear_relu_stack=nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10),
        )
        
    def forward(self,x):
        x=self.flatten(x)
        logits=self.linear_relu_stack(x)
        return logits

In [14]:
model=NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [19]:
import numpy as np
X=torch.rand(1,28,28,device=device)
logits=model(X)
logits_np=logits.cpu().detach().numpy()
pred_probab=nn.Softmax(dim=1)(logits)# 这里的dim=1意味着是进行纵向对比，
pred_probab_np=pred_probab.cpu().detach().numpy()
# Softmax 对向量中的每一个元素 $z_i$ 执行以下计算：$$P_i = \frac{e^{z_i}}{\sum_{j=1}^{10} e^{z_j}}$$指数化 ($e^{z_i}$)：
# 不管原始分正负，经过 $e$ 的次方后全部变成正数。而且大分会变得极大，小分会变得极小（拉开差距）
# 归一化 ($\sum$)：除以所有分数的总和，确保结果在 $0$ 到 $1$ 之间，且所有类别概率之和为 1。
y_pred=pred_probab.argmax(1)
print(f"Logits : {logits_np}")         
print(f"Probabilities : {pred_probab_np}") 
print(f"Prediction index: {y_pred}")            

Logits : [[ 0.00789311  0.05726685  0.00605401  0.02869887  0.02696147 -0.06251951
  -0.12975156 -0.05790151 -0.07860193 -0.01453054]]
Probabilities : [[0.10284092 0.108046   0.10265195 0.10500301 0.10482074 0.09584868
  0.08961642 0.09629233 0.09431953 0.10056051]]
Prediction index: tensor([1])


In [20]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [26]:
flatten=nn.Flatten()
flat_image=flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [27]:
layer1 =nn.Linear(in_features=28*28,out_features=20)
hidden1=layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [23]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1128,  0.0418,  0.0373, -0.1730, -0.1284, -0.4047, -0.5356, -0.2045,
          0.0906,  0.0420,  0.2900, -0.4506,  0.2668,  0.2209, -0.2769,  0.2248,
         -0.1346,  0.0248,  0.6326, -0.0836],
        [-0.2154,  0.0279,  0.1900, -0.0509, -0.2889, -0.1596, -0.2467,  0.0729,
          0.2699, -0.1213,  0.6991, -0.6987,  0.6300,  0.1314, -0.1149,  0.3381,
         -0.0289, -0.3341,  1.0240,  0.0444],
        [-0.0918, -0.0563,  0.0050,  0.0636, -0.1293, -0.2488, -0.7078, -0.1330,
         -0.0591, -0.2139,  0.6217, -0.5197,  0.3601,  0.2951, -0.3224,  0.3329,
         -0.0114, -0.1301,  0.7968, -0.2309]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.0418, 0.0373, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0906,
         0.0420, 0.2900, 0.0000, 0.2668, 0.2209, 0.0000, 0.2248, 0.0000, 0.0248,
         0.6326, 0.0000],
        [0.0000, 0.0279, 0.1900, 0.0000, 0.0000, 0.0000, 0.0000, 0.0729, 0.2699,
         0.0000, 0.6991, 0.0000, 0.6300, 0.1314, 0.00

In [24]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [29]:
softmax=nn.Softmax(dim=1)
pred_probab=softmax(logits)
print(f"Model structure: {model}\n\n")

for name,param in model.named_parameters():#name字符串，param是tensor对象
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0218, -0.0167,  0.0186,  ..., -0.0038, -0.0053,  0.0352],
        [-0.0018, -0.0185,  0.0221,  ...,  0.0236,  0.0047, -0.0009]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0132, 0.0085], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0069,  0.0361, -0.0021,  ..., -0.0198,  0.0068,  0.0132],
        [ 0.0337,  0.0206,  0.0215,  ...,  0.0034,  0.0075, -0.0081]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si